# Notebook 25 — Constraint Budget Allocation

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 24 evaluated route-memory policies using stability, regret, fallback reduction, and constraint alignment.

Notebook 25 adds a finite routing budget:

- reroute budget,
- decompression budget,
- fallback penalty,
- pressure-aware allocation,
- policy regret under resource constraints.

Constraint view:
> predictive routing is useful only when finite constraint budgets are allocated before fallback pressure dominates.

## Goals

1. Load Notebook 24 policy evaluation outputs when available.
2. Construct budget scenarios over route-memory policy decisions.
3. Allocate finite routing resources across windows.
4. Compare allocation strategies:
   - uniform budget,
   - pressure-first budget,
   - forecast-first budget,
   - CGCS-balanced budget,
   - regret-minimizing budget.
5. Export CSV, JSON, Markdown report, and PNG figures.
6. Generate a Colab-downloadable output zip.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load Notebook 24 outputs

Preferred input:

```text
results/notebook24_route_memory_policy_evaluation.csv
```

If unavailable, this notebook creates a fallback policy-evaluation stream.

In [ ]:
input_path = RESULTS_DIR / "notebook24_route_memory_policy_evaluation.csv"

if input_path.exists():
    eval_df = pd.read_csv(input_path)
    print("Loaded:", input_path)
else:
    print("Notebook 24 output not found; creating fallback stream.")
    rng = np.random.default_rng(25)
    n = 240
    policies = ["reactive", "predictive", "conservative_predictive", "aggressive_predictive", "cgcs_balanced"]
    macro_routes = ["macro_0", "macro_1", "macro_2", "macro_3", "macro_4"]
    rows = []
    for trial in range(30):
        for policy in policies:
            for w in range(n):
                pressure = np.clip(0.5 + 0.25*np.sin(w/18) + rng.normal(0, 0.12), 0, 1)
                risk = np.clip(0.35 + 0.35*pressure + rng.normal(0, 0.18), 0, 1)
                macro = rng.choice(macro_routes)
                if policy == "reactive":
                    gate = "fallback" if risk > 0.82 else "watch"
                elif policy == "conservative_predictive":
                    gate = "reroute" if risk > 0.62 else ("fallback" if risk > 0.88 else "watch")
                elif policy == "aggressive_predictive":
                    gate = "reroute" if risk > 0.40 else ("fallback" if risk > 0.78 else "watch")
                elif policy == "cgcs_balanced":
                    gate = "reroute" if risk > 0.55 and pressure > 0.45 else ("fallback" if risk > 0.84 else "watch")
                else:
                    gate = "reroute" if risk > 0.50 else ("fallback" if risk > 0.80 else "watch")
                stability = np.clip(0.55 + 0.12*(gate=="reroute") - 0.16*(gate=="fallback") - 0.10*pressure + rng.normal(0,0.03),0,1)
                constraint_score = np.clip(0.45*stability + 0.25*(1-pressure) + 0.20*(1-risk) + 0.10*(gate!="fallback"),0,1)
                cost = 1.0*(gate=="fallback") + 0.35*(gate=="reroute") + 0.15*(gate=="watch") + 0.20*pressure - 0.25*stability
                rows.append({
                    "trial": trial,
                    "window_id": w,
                    "policy": policy,
                    "macro_route": macro,
                    "gate": gate,
                    "risk": risk,
                    "pressure": pressure,
                    "stability": stability,
                    "constraint_score": constraint_score,
                    "policy_cost": cost,
                    "regret": max(cost - 0.25, 0),
                    "future_decompression": int(risk > 0.70),
                })
    eval_df = pd.DataFrame(rows)
    eval_df["switch_rate"] = eval_df.groupby(["trial","policy"])["gate"].transform(lambda s: s.ne(s.shift()).rolling(15, min_periods=1).mean())

# Normalize expected columns.
if "window_id" not in eval_df.columns:
    eval_df["window_id"] = eval_df.get("window", pd.Series(np.arange(len(eval_df))))
if "trial" not in eval_df.columns:
    eval_df["trial"] = 0
if "policy" not in eval_df.columns:
    eval_df["policy"] = "unknown_policy"
if "macro_route" not in eval_df.columns:
    eval_df["macro_route"] = "macro_unknown"
if "gate" not in eval_df.columns:
    eval_df["gate"] = "watch"

for c in ["risk", "pressure", "stability", "constraint_score", "policy_cost", "regret", "switch_rate"]:
    if c not in eval_df.columns:
        eval_df[c] = 0.5
    eval_df[c] = pd.to_numeric(eval_df[c], errors="coerce").fillna(0.5).clip(lower=0)

if "future_decompression" not in eval_df.columns:
    eval_df["future_decompression"] = ((eval_df["risk"] > 0.65) | (eval_df["gate"] == "fallback")).astype(int)

eval_df.head()

## Budget allocation model

Each strategy assigns finite resource units:

- `reroute_units`
- `decompression_units`
- `monitor_units`

Allocating resources can reduce expected fallback cost, but resource overuse produces budget regret.

In [ ]:
budget_strategies = {
    "uniform_budget": {
        "reroute_budget_rate": 0.30,
        "decompression_budget_rate": 0.18,
        "monitor_budget_rate": 0.60,
        "pressure_weight": 0.20,
        "risk_weight": 0.20,
        "regret_weight": 0.20,
        "constraint_weight": 0.20,
    },
    "pressure_first": {
        "reroute_budget_rate": 0.34,
        "decompression_budget_rate": 0.22,
        "monitor_budget_rate": 0.52,
        "pressure_weight": 0.45,
        "risk_weight": 0.20,
        "regret_weight": 0.15,
        "constraint_weight": 0.10,
    },
    "forecast_first": {
        "reroute_budget_rate": 0.38,
        "decompression_budget_rate": 0.25,
        "monitor_budget_rate": 0.50,
        "pressure_weight": 0.15,
        "risk_weight": 0.45,
        "regret_weight": 0.15,
        "constraint_weight": 0.15,
    },
    "cgcs_balanced_budget": {
        "reroute_budget_rate": 0.32,
        "decompression_budget_rate": 0.22,
        "monitor_budget_rate": 0.58,
        "pressure_weight": 0.25,
        "risk_weight": 0.25,
        "regret_weight": 0.20,
        "constraint_weight": 0.25,
    },
    "regret_minimizing_budget": {
        "reroute_budget_rate": 0.28,
        "decompression_budget_rate": 0.20,
        "monitor_budget_rate": 0.55,
        "pressure_weight": 0.15,
        "risk_weight": 0.20,
        "regret_weight": 0.45,
        "constraint_weight": 0.10,
    },
}

def allocate_budget(df, strategy_name, cfg):
    rows = []
    for (trial, policy), part in df.groupby(["trial", "policy"]):
        part = part.sort_values("window_id").copy()
        n = len(part)
        reroute_budget = max(1, int(np.ceil(cfg["reroute_budget_rate"] * n)))
        decompression_budget = max(1, int(np.ceil(cfg["decompression_budget_rate"] * n)))
        monitor_budget = max(1, int(np.ceil(cfg["monitor_budget_rate"] * n)))

        # Priority score: high means allocate sooner.
        regret_norm = part["regret"] / max(part["regret"].max(), 1e-9)
        constraint_gap = 1 - part["constraint_score"].clip(0, 1)
        priority = (
            cfg["pressure_weight"] * part["pressure"].clip(0, 1)
            + cfg["risk_weight"] * part["risk"].clip(0, 1)
            + cfg["regret_weight"] * regret_norm
            + cfg["constraint_weight"] * constraint_gap
        )
        part["budget_priority"] = priority

        reroute_candidates = part[part["gate"].isin(["reroute", "fallback"])].sort_values("budget_priority", ascending=False)
        decomp_candidates = part[(part["future_decompression"] == 1) | (part["gate"] == "fallback")].sort_values("budget_priority", ascending=False)
        monitor_candidates = part[part["gate"].isin(["watch", "fallback"])].sort_values("budget_priority", ascending=False)

        reroute_alloc = set(reroute_candidates.head(reroute_budget).index)
        decomp_alloc = set(decomp_candidates.head(decompression_budget).index)
        monitor_alloc = set(monitor_candidates.head(monitor_budget).index)

        for idx, row in part.iterrows():
            allocated_reroute = idx in reroute_alloc
            allocated_decomp = idx in decomp_alloc
            allocated_monitor = idx in monitor_alloc

            allocation_count = int(allocated_reroute) + int(allocated_decomp) + int(allocated_monitor)
            avoided_fallback = int(row["gate"] == "fallback" and (allocated_reroute or allocated_decomp))
            stabilized_watch = int(row["gate"] == "watch" and allocated_monitor)

            adjusted_cost = float(row["policy_cost"])
            adjusted_cost -= 0.32 * avoided_fallback
            adjusted_cost -= 0.18 * int(allocated_reroute and row["risk"] > 0.50)
            adjusted_cost -= 0.20 * int(allocated_decomp and row["future_decompression"] == 1)
            adjusted_cost -= 0.06 * stabilized_watch
            adjusted_cost += 0.04 * allocation_count

            adjusted_stability = np.clip(
                float(row["stability"])
                + 0.10 * avoided_fallback
                + 0.06 * int(allocated_reroute)
                + 0.07 * int(allocated_decomp)
                + 0.03 * stabilized_watch
                - 0.02 * max(allocation_count - 2, 0),
                0,
                1,
            )

            adjusted_constraint = np.clip(
                0.45 * adjusted_stability
                + 0.25 * (1 - float(row["pressure"]))
                + 0.20 * float(row["constraint_score"])
                + 0.10 * int(allocation_count > 0),
                0,
                1,
            )

            rows.append({
                "trial": int(trial),
                "window_id": int(row["window_id"]),
                "policy": policy,
                "budget_strategy": strategy_name,
                "macro_route": row["macro_route"],
                "gate": row["gate"],
                "risk": float(row["risk"]),
                "pressure": float(row["pressure"]),
                "budget_priority": float(row["budget_priority"]),
                "allocated_reroute": int(allocated_reroute),
                "allocated_decompression": int(allocated_decomp),
                "allocated_monitor": int(allocated_monitor),
                "allocation_count": allocation_count,
                "avoided_fallback": avoided_fallback,
                "stabilized_watch": stabilized_watch,
                "original_cost": float(row["policy_cost"]),
                "adjusted_cost": float(max(adjusted_cost, 0)),
                "original_stability": float(row["stability"]),
                "adjusted_stability": float(adjusted_stability),
                "original_constraint_score": float(row["constraint_score"]),
                "adjusted_constraint_score": float(adjusted_constraint),
                "future_decompression": int(row["future_decompression"]),
            })
    out = pd.DataFrame(rows)
    out["budget_regret"] = out["adjusted_cost"] - out.groupby(["trial","window_id"])["adjusted_cost"].transform("min")
    out["budget_switch"] = out.groupby(["trial","policy","budget_strategy"])["gate"].transform(lambda s: s.ne(s.shift()).astype(int))
    out["budget_switch_rate"] = out.groupby(["trial","policy","budget_strategy"])["budget_switch"].transform(lambda s: s.rolling(15, min_periods=1).mean())
    return out

budget_frames = []
for name, cfg in budget_strategies.items():
    budget_frames.append(allocate_budget(eval_df, name, cfg))

budget_df = pd.concat(budget_frames, ignore_index=True)
budget_df.head()

## Summary tables

In [ ]:
budget_summary = (
    budget_df.groupby("budget_strategy")
    .agg(
        windows=("window_id", "count"),
        mean_adjusted_constraint_score=("adjusted_constraint_score", "mean"),
        mean_adjusted_stability=("adjusted_stability", "mean"),
        mean_adjusted_cost=("adjusted_cost", "mean"),
        mean_budget_regret=("budget_regret", "mean"),
        reroute_allocation_rate=("allocated_reroute", "mean"),
        decompression_allocation_rate=("allocated_decompression", "mean"),
        monitor_allocation_rate=("allocated_monitor", "mean"),
        avoided_fallback_rate=("avoided_fallback", "mean"),
        mean_budget_switch_rate=("budget_switch_rate", "mean"),
    )
    .reset_index()
    .sort_values("mean_adjusted_constraint_score", ascending=False)
)

policy_budget_summary = (
    budget_df.groupby(["policy", "budget_strategy"])
    .agg(
        windows=("window_id", "count"),
        mean_adjusted_constraint_score=("adjusted_constraint_score", "mean"),
        mean_adjusted_stability=("adjusted_stability", "mean"),
        mean_adjusted_cost=("adjusted_cost", "mean"),
        mean_budget_regret=("budget_regret", "mean"),
        avoided_fallback_rate=("avoided_fallback", "mean"),
        reroute_allocation_rate=("allocated_reroute", "mean"),
        decompression_allocation_rate=("allocated_decompression", "mean"),
    )
    .reset_index()
)

route_budget_summary = (
    budget_df.groupby(["macro_route", "budget_strategy"])
    .agg(
        windows=("window_id", "count"),
        mean_adjusted_constraint_score=("adjusted_constraint_score", "mean"),
        mean_adjusted_stability=("adjusted_stability", "mean"),
        mean_budget_regret=("budget_regret", "mean"),
        avoided_fallback_rate=("avoided_fallback", "mean"),
        allocation_rate=("allocation_count", lambda s: (s > 0).mean()),
    )
    .reset_index()
)

budget_summary

## Export results

In [ ]:
budget_csv = RESULTS_DIR / "notebook25_constraint_budget_allocation.csv"
budget_json = RESULTS_DIR / "notebook25_constraint_budget_allocation.json"
budget_summary_csv = RESULTS_DIR / "notebook25_budget_summary.csv"
policy_budget_summary_csv = RESULTS_DIR / "notebook25_policy_budget_summary.csv"
route_budget_summary_csv = RESULTS_DIR / "notebook25_route_budget_summary.csv"

budget_df.to_csv(budget_csv, index=False)
budget_df.to_json(budget_json, orient="records", indent=2)
budget_summary.to_csv(budget_summary_csv, index=False)
policy_budget_summary.to_csv(policy_budget_summary_csv, index=False)
route_budget_summary.to_csv(route_budget_summary_csv, index=False)

print("Saved:", budget_csv)
print("Saved:", budget_json)
print("Saved:", budget_summary_csv)
print("Saved:", policy_budget_summary_csv)
print("Saved:", route_budget_summary_csv)

## Figure 1 — Budget recommendation score

In [ ]:
budget_recommendation = budget_summary.copy()
budget_recommendation["recommendation_score"] = (
    0.35 * budget_recommendation["mean_adjusted_constraint_score"]
    + 0.25 * budget_recommendation["mean_adjusted_stability"]
    + 0.20 * (1 - budget_recommendation["mean_budget_regret"] / max(budget_recommendation["mean_budget_regret"].max(), 1e-9))
    + 0.20 * budget_recommendation["avoided_fallback_rate"]
)
budget_recommendation = budget_recommendation.sort_values("recommendation_score")

budget_recommendation_fig = FIGURES_DIR / "notebook25_budget_recommendation_score.png"

plt.figure(figsize=(10, 6))
plt.bar(budget_recommendation["budget_strategy"], budget_recommendation["recommendation_score"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Recommendation score")
plt.title("Constraint Budget Allocation: Budget Recommendation Score")
plt.tight_layout()
plt.savefig(budget_recommendation_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", budget_recommendation_fig)

## Figure 2 — Adjusted constraint score

In [ ]:
constraint_fig = FIGURES_DIR / "notebook25_adjusted_constraint_score.png"

plot_df = budget_summary.sort_values("mean_adjusted_constraint_score")

plt.figure(figsize=(10, 6))
plt.bar(plot_df["budget_strategy"], plot_df["mean_adjusted_constraint_score"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Mean adjusted constraint score")
plt.title("Constraint Budget Allocation: Adjusted Constraint Score")
plt.tight_layout()
plt.savefig(constraint_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", constraint_fig)

## Figure 3 — Budget regret

In [ ]:
regret_fig = FIGURES_DIR / "notebook25_budget_regret.png"

plot_df = budget_summary.sort_values("mean_budget_regret")

plt.figure(figsize=(10, 6))
plt.bar(plot_df["budget_strategy"], plot_df["mean_budget_regret"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Mean budget regret")
plt.title("Constraint Budget Allocation: Mean Budget Regret")
plt.tight_layout()
plt.savefig(regret_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", regret_fig)

## Figure 4 — Allocation rates

In [ ]:
allocation_rates_fig = FIGURES_DIR / "notebook25_allocation_rates.png"

plot_df = budget_summary.sort_values("budget_strategy")
x = np.arange(len(plot_df))
width = 0.25

plt.figure(figsize=(11, 6))
plt.bar(x - width, plot_df["reroute_allocation_rate"], width, label="reroute")
plt.bar(x, plot_df["decompression_allocation_rate"], width, label="decompression")
plt.bar(x + width, plot_df["monitor_allocation_rate"], width, label="monitor")
plt.xticks(x, plot_df["budget_strategy"], rotation=35, ha="right")
plt.ylabel("Allocation rate")
plt.title("Constraint Budget Allocation: Allocation Rates")
plt.legend()
plt.tight_layout()
plt.savefig(allocation_rates_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", allocation_rates_fig)

## Figure 5 — Avoided fallback

In [ ]:
avoided_fallback_fig = FIGURES_DIR / "notebook25_avoided_fallback_rate.png"

plot_df = budget_summary.sort_values("avoided_fallback_rate")

plt.figure(figsize=(10, 6))
plt.bar(plot_df["budget_strategy"], plot_df["avoided_fallback_rate"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Avoided fallback rate")
plt.title("Constraint Budget Allocation: Avoided Fallback Rate")
plt.tight_layout()
plt.savefig(avoided_fallback_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", avoided_fallback_fig)

## Figure 6 — Policy × budget heatmap

In [ ]:
policy_budget_heatmap_fig = FIGURES_DIR / "notebook25_policy_budget_constraint_heatmap.png"

heat = policy_budget_summary.pivot(
    index="policy",
    columns="budget_strategy",
    values="mean_adjusted_constraint_score",
).fillna(0)

plt.figure(figsize=(11, 6))
plt.imshow(heat.values, aspect="auto", vmin=0, vmax=1)
plt.xticks(range(len(heat.columns)), heat.columns, rotation=35, ha="right")
plt.yticks(range(len(heat.index)), heat.index)
plt.colorbar(label="Mean adjusted constraint score")
plt.title("Constraint Budget Allocation: Policy × Budget Constraint Score")
plt.tight_layout()
plt.savefig(policy_budget_heatmap_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", policy_budget_heatmap_fig)

## Figure 7 — Route × budget heatmap

In [ ]:
route_budget_heatmap_fig = FIGURES_DIR / "notebook25_route_budget_constraint_heatmap.png"

heat = route_budget_summary.pivot(
    index="macro_route",
    columns="budget_strategy",
    values="mean_adjusted_constraint_score",
).fillna(0)

plt.figure(figsize=(11, 6))
plt.imshow(heat.values, aspect="auto", vmin=0, vmax=1)
plt.xticks(range(len(heat.columns)), heat.columns, rotation=35, ha="right")
plt.yticks(range(len(heat.index)), heat.index)
plt.colorbar(label="Mean adjusted constraint score")
plt.title("Constraint Budget Allocation: Route × Budget Constraint Score")
plt.tight_layout()
plt.savefig(route_budget_heatmap_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", route_budget_heatmap_fig)

## Figure 8 — Budget priority timeline

In [ ]:
priority_fig = FIGURES_DIR / "notebook25_budget_priority_timeline.png"

sample = budget_df[
    (budget_df["trial"] == budget_df["trial"].min())
    & (budget_df["policy"] == "cgcs_balanced")
].copy()

if sample.empty:
    sample = budget_df[budget_df["trial"] == budget_df["trial"].min()].copy()

plt.figure(figsize=(14, 6))
for strategy in sorted(sample["budget_strategy"].unique()):
    part = sample[sample["budget_strategy"] == strategy].sort_values("window_id")
    plt.plot(part["window_id"], part["budget_priority"], label=strategy)
plt.xlabel("Window")
plt.ylabel("Budget priority")
plt.title("Constraint Budget Allocation: Budget Priority Timeline")
plt.legend()
plt.tight_layout()
plt.savefig(priority_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", priority_fig)

## Figure 9 — Adjusted cost timeline

In [ ]:
cost_timeline_fig = FIGURES_DIR / "notebook25_adjusted_cost_timeline.png"

sample = budget_df[
    (budget_df["trial"] == budget_df["trial"].min())
    & (budget_df["policy"] == "cgcs_balanced")
].copy()

if sample.empty:
    sample = budget_df[budget_df["trial"] == budget_df["trial"].min()].copy()

plt.figure(figsize=(14, 6))
for strategy in sorted(sample["budget_strategy"].unique()):
    part = sample[sample["budget_strategy"] == strategy].sort_values("window_id")
    rolling_cost = part["adjusted_cost"].rolling(10, min_periods=1).mean()
    plt.plot(part["window_id"], rolling_cost, label=strategy)
plt.xlabel("Window")
plt.ylabel("Rolling adjusted cost")
plt.title("Constraint Budget Allocation: Adjusted Cost Timeline")
plt.legend()
plt.tight_layout()
plt.savefig(cost_timeline_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", cost_timeline_fig)

## Figure 10 — Budget frontier

In [ ]:
frontier_fig = FIGURES_DIR / "notebook25_budget_frontier.png"

plot_df = budget_summary.copy()

plt.figure(figsize=(8, 6))
plt.scatter(plot_df["mean_budget_regret"], plot_df["mean_adjusted_constraint_score"], s=120)
for _, row in plot_df.iterrows():
    plt.annotate(row["budget_strategy"], (row["mean_budget_regret"], row["mean_adjusted_constraint_score"]))
plt.xlabel("Mean budget regret (lower is better)")
plt.ylabel("Mean adjusted constraint score (higher is better)")
plt.title("Constraint Budget Allocation: Regret / Constraint Frontier")
plt.tight_layout()
plt.savefig(frontier_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", frontier_fig)

## Full Markdown report

In [ ]:
report_path = REPORTS_DIR / "report_25_constraint_budget_allocation.md"

links = {
    "budget_csv": "results/notebook25_constraint_budget_allocation.csv",
    "budget_json": "results/notebook25_constraint_budget_allocation.json",
    "budget_summary_csv": "results/notebook25_budget_summary.csv",
    "policy_budget_summary_csv": "results/notebook25_policy_budget_summary.csv",
    "route_budget_summary_csv": "results/notebook25_route_budget_summary.csv",
    "budget_recommendation_fig": "figures/notebook25_budget_recommendation_score.png",
    "constraint_fig": "figures/notebook25_adjusted_constraint_score.png",
    "regret_fig": "figures/notebook25_budget_regret.png",
    "allocation_rates_fig": "figures/notebook25_allocation_rates.png",
    "avoided_fallback_fig": "figures/notebook25_avoided_fallback_rate.png",
    "policy_budget_heatmap_fig": "figures/notebook25_policy_budget_constraint_heatmap.png",
    "route_budget_heatmap_fig": "figures/notebook25_route_budget_constraint_heatmap.png",
    "priority_fig": "figures/notebook25_budget_priority_timeline.png",
    "cost_timeline_fig": "figures/notebook25_adjusted_cost_timeline.png",
    "frontier_fig": "figures/notebook25_budget_frontier.png",
}

best_budget = budget_recommendation.sort_values("recommendation_score", ascending=False).iloc[0]["budget_strategy"]

report_lines = [
    "# Report 25 — Constraint Budget Allocation",
    "",
    "This report adds finite resource allocation to predictive route-memory policy evaluation.",
    "",
    "Constraint view:",
    "> predictive routing is useful only when finite constraint budgets are allocated before fallback pressure dominates.",
    "",
    "## Generated outputs",
    "",
    f'- Budget allocation CSV: <a href="{links["budget_csv"]}">`{links["budget_csv"]}`</a>',
    f'- Budget allocation JSON: <a href="{links["budget_json"]}">`{links["budget_json"]}`</a>',
    f'- Budget summary CSV: <a href="{links["budget_summary_csv"]}">`{links["budget_summary_csv"]}`</a>',
    f'- Policy-budget summary CSV: <a href="{links["policy_budget_summary_csv"]}">`{links["policy_budget_summary_csv"]}`</a>',
    f'- Route-budget summary CSV: <a href="{links["route_budget_summary_csv"]}">`{links["route_budget_summary_csv"]}`</a>',
    f'- Figure: <a href="{links["budget_recommendation_fig"]}">`{links["budget_recommendation_fig"]}`</a>',
    f'- Figure: <a href="{links["constraint_fig"]}">`{links["constraint_fig"]}`</a>',
    f'- Figure: <a href="{links["regret_fig"]}">`{links["regret_fig"]}`</a>',
    f'- Figure: <a href="{links["allocation_rates_fig"]}">`{links["allocation_rates_fig"]}`</a>',
    f'- Figure: <a href="{links["avoided_fallback_fig"]}">`{links["avoided_fallback_fig"]}`</a>',
    f'- Figure: <a href="{links["policy_budget_heatmap_fig"]}">`{links["policy_budget_heatmap_fig"]}`</a>',
    f'- Figure: <a href="{links["route_budget_heatmap_fig"]}">`{links["route_budget_heatmap_fig"]}`</a>',
    f'- Figure: <a href="{links["priority_fig"]}">`{links["priority_fig"]}`</a>',
    f'- Figure: <a href="{links["cost_timeline_fig"]}">`{links["cost_timeline_fig"]}`</a>',
    f'- Figure: <a href="{links["frontier_fig"]}">`{links["frontier_fig"]}`</a>',
    "",
    "## Budget strategy summary",
    "",
    budget_summary.to_markdown(index=False),
    "",
    "## Policy-budget summary",
    "",
    policy_budget_summary.to_markdown(index=False),
    "",
    "## Route-budget summary",
    "",
    route_budget_summary.to_markdown(index=False),
    "",
    "## Recommendation summary",
    "",
    budget_recommendation.sort_values("recommendation_score", ascending=False).to_markdown(index=False),
    "",
    "## Interpretation",
    "",
    "- Budget allocation turns predictive decompression into a finite-resource routing problem.",
    "- Pressure-first allocation prioritizes high-pressure windows, but can overspend on noisy pressure spikes.",
    "- Forecast-first allocation prioritizes expected decompression, but can miss low-probability high-cost collapse.",
    "- CGCS-balanced allocation spreads resources across pressure, forecast risk, regret, and constraint score.",
    "- Regret-minimizing allocation learns from policy cost but may under-allocate to early weak signals.",
    f"- Best budget strategy by recommendation score in this run: `{best_budget}`.",
    "",
    "## Next step",
    "",
    "Notebook 26 can build budget phase diagrams:",
    "- sweep reroute budget,",
    "- sweep decompression budget,",
    "- vary fallback penalty,",
    "- identify stable budget regimes and failure boundaries.",
]

report_path.write_text("\n".join(report_lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if running in Google Colab.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook25_constraint_budget_allocation_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook25_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_25_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))